# ComfyUI on Google Colab v5e-1 TPU
Select **Runtime → Change runtime type → TPU (v5e-1)** before running. Cache import/export is local and portable; Google Drive is optional only for models.


In [ ]:
import os, subprocess
assert os.environ.get('COLAB_TPU_ADDR') or os.environ.get('TPU_NAME') or os.path.exists('/dev/accel0'), 'Select a TPU runtime before continuing'
print('TPU runtime detected')


## Install the TPU runtime
The official PyTorch/XLA TPU package index resolves a mutually compatible runtime. Re-running skips packages that are already importable.


In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec('torch_xla') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'torch_xla[tpu]', '-f', 'https://storage.googleapis.com/libtpu-releases/index.html'])


In [ ]:
from pathlib import Path
REPO_URL = 'https://github.com/YOUR_GITHUB_USER/ComfyUi-TPU.git'  # Set your fork URL.
BRANCH = 'tpu-v5e'
ROOT = Path('/content/ComfyUi-TPU')
if not (ROOT / '.git').exists():
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, REPO_URL, str(ROOT)])
else:
    subprocess.check_call(['git', '-C', str(ROOT), 'pull', '--ff-only'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT / 'requirements.txt')])
os.chdir(ROOT)


## Optional cache import
Run this before any TPU operation. Cancel the upload dialog to start with an empty cache.


In [ ]:
from google.colab import files
uploaded = files.upload()
for name in uploaded:
    if name.endswith('.zip'):
        subprocess.check_call([sys.executable, 'tools/tpu_cache.py', 'import', name])
        break


## Capability probe
Review every operation. An `error` is an unsupported path, not a passed test.


In [ ]:
subprocess.check_call([sys.executable, 'tools/tpu_probe.py'])


## Optional model storage
Local `/content` paths work directly. Mount Drive only if desired, then configure its directories with `extra_model_paths.yaml`. Hugging Face downloads should be explicitly initiated by the user.


In [ ]:
# Optional:
# from google.colab import drive
# drive.mount('/content/drive')


## Launch and tunnel
This downloads Cloudflare's tunnel binary and starts ComfyUI. The printed `trycloudflare.com` URL is the remote UI.


In [ ]:
import re, time, urllib.request
cloudflared = Path('/content/cloudflared')
if not cloudflared.exists():
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', cloudflared)
    cloudflared.chmod(0o755)
server = subprocess.Popen([sys.executable, 'main.py', '--tpu', '--listen', '0.0.0.0'], cwd=ROOT)
time.sleep(10)
tunnel = subprocess.Popen([str(cloudflared), 'tunnel', '--url', 'http://127.0.0.1:8188'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in tunnel.stdout:
    print(line, end='')
    match = re.search(r'https://[^ ]+\.trycloudflare\.com', line)
    if match:
        print('ComfyUI URL:', match.group(0)); break


## Export cache to your computer


In [ ]:
archive = ROOT / 'comfyui-v5e-tpu-cache.zip'
subprocess.check_call([sys.executable, 'tools/tpu_cache.py', 'export', str(archive)])
files.download(str(archive))
